# RCV1 — Initial Exploratory Analysis

Standalone notebook to check that the download/setup step (`download_rcv1.py`)
completed correctly and to get a first look at the dataset. No Dask here —
everything runs in-memory with NumPy/SciPy, since the data comfortably fits
in RAM as sparse matrices.

Assumes `download_rcv1.py` has already been run and produced `meta.json`,
`rcv1_X.npz`, `rcv1_y.npz` in the processed data folder.

In [ ]:
import json
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt


## 1. Configuration

Set this to the exact folder used as `processed_dir` in `download_rcv1.py`.

In [ ]:
PROCESSED_DIR = Path("/home/ubuntu/Project/datasets/rcv1")


## 2. Load metadata and verify the download

Checks that `meta.json` exists and that the `.npz` files it points to are
actually present on disk.

In [ ]:
meta_path = PROCESSED_DIR / "meta.json"

if not meta_path.exists():
    raise FileNotFoundError(
        f"{meta_path} not found. Run download_rcv1.py first."
    )

with open(meta_path) as f:
    meta = json.load(f)

meta


In [ ]:
x_path = Path(meta["x_path"])
y_path = Path(meta["y_path"])

assert x_path.exists(), f"Missing file: {x_path}"
assert y_path.exists(), f"Missing file: {y_path}"

print(f"X file: {x_path} ({x_path.stat().st_size / 1024**2:.1f} MB)")
print(f"y file: {y_path} ({y_path.stat().st_size / 1024**2:.1f} MB)")
print("Both files found on disk.")


## 3. Load the data and cross-check against `meta.json`

Loading directly confirms the `.npz` files are not corrupted, and comparing
shapes/nnz against the recorded metadata confirms the download completed
correctly (no missing or truncated data).

In [ ]:
X = sp.load_npz(x_path)
y = sp.load_npz(y_path)

print(f"X: shape={X.shape}, nnz={X.nnz:,}, dtype={X.dtype}")
print(f"y: shape={y.shape}, nnz={y.nnz:,}, dtype={y.dtype}")

assert X.shape == (meta["n_samples"], meta["n_features"]), "X shape mismatch vs meta.json"
assert y.shape == (meta["n_samples"], meta["n_categories"]), "y shape mismatch vs meta.json"
assert X.nnz == meta["x_nnz"], "X nnz mismatch vs meta.json"
assert y.nnz == meta["y_nnz"], "y nnz mismatch vs meta.json"

print("\nAll checks passed: the download and save completed correctly.")


## 4. Exploratory Data Analysis

### 4.1 Non-zero features per document

How many distinct terms (non-zero TF-IDF entries) each document has.

In [ ]:
nnz_per_doc = X.getnnz(axis=1)

print(f"Min:    {nnz_per_doc.min()}")
print(f"Max:    {nnz_per_doc.max()}")
print(f"Mean:   {nnz_per_doc.mean():.1f}")
print(f"Median: {np.median(nnz_per_doc):.1f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(nnz_per_doc, bins=100)
ax.set_xlabel("Non-zero features per document")
ax.set_ylabel("Number of documents")
ax.set_title("Distribution of document sparsity")
plt.show()


### 4.2 Distribution of non-zero TF-IDF values

Sampled for speed, since `X.data` has ~130M entries.

In [ ]:
rng = np.random.default_rng(42)
sample_size = min(500_000, X.data.size)
sample = rng.choice(X.data, size=sample_size, replace=False)

print(f"Min:    {sample.min():.4f}")
print(f"Max:    {sample.max():.4f}")
print(f"Mean:   {sample.mean():.4f}")
print(f"Median: {np.median(sample):.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sample, bins=100)
ax.set_xlabel("Non-zero TF-IDF value")
ax.set_ylabel("Frequency (sampled)")
ax.set_title(f"Distribution of non-zero values (sample of {sample_size:,})")
plt.show()


### 4.3 Category (label) distribution

RCV1 is known for having a strongly imbalanced label distribution. This
doesn't affect k-means directly (which will use `X` only, not `y`), but it's
useful context if the labels are later used as ground truth to validate the
clustering (e.g. purity, NMI, ARI).

In [ ]:
docs_per_category = np.asarray(y.sum(axis=0)).ravel()

print(f"Categories: {len(docs_per_category)}")
print(f"Min docs/category:    {docs_per_category.min()}")
print(f"Max docs/category:    {docs_per_category.max()}")
print(f"Median docs/category: {np.median(docs_per_category):.0f}")

order = np.argsort(docs_per_category)[::-1]
top_n = 15

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(top_n), docs_per_category[order[:top_n]])
ax.set_xlabel("Category rank")
ax.set_ylabel("Number of documents")
ax.set_title(f"Top {top_n} most frequent categories")
plt.show()


### 4.4 Labels per document (multilabel cardinality)

Since RCV1 is multilabel, each document can belong to more than one
category.

In [ ]:
labels_per_doc = y.getnnz(axis=1)

print(f"Min:    {labels_per_doc.min()}")
print(f"Max:    {labels_per_doc.max()}")
print(f"Mean:   {labels_per_doc.mean():.2f}")
print(f"Median: {np.median(labels_per_doc):.0f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(labels_per_doc, bins=range(0, labels_per_doc.max() + 2))
ax.set_xlabel("Number of labels per document")
ax.set_ylabel("Number of documents")
ax.set_title("Multilabel cardinality distribution")
plt.show()


## Takeaways

A short summary to fill in after looking at the plots above — e.g. whether
document length is fairly uniform or heavy-tailed, and how imbalanced the
categories are. This context will inform choices made later in the Dask /
benchmarking notebook (e.g. chunk size).